## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(tf.random.normal([784, 256]), stddev = 0.1)
        self.b1 = tf.Variable(tf.zeros([256]))
        self.W2 = tf.Variable(tf.random.normal([256, 10]), stddev = 0.1)
        self.b2 = tf.Variable(tf.zeros([10]))
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        x = tf.reshape(x, [-1, 784])
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [7]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 23.35599 ; accuracy 0.57166666
epoch 1 : loss 23.19763 ; accuracy 0.5736833
epoch 2 : loss 23.04216 ; accuracy 0.57593334
epoch 3 : loss 22.889503 ; accuracy 0.57778335
epoch 4 : loss 22.7396 ; accuracy 0.5796
epoch 5 : loss 22.592405 ; accuracy 0.5816333
epoch 6 : loss 22.4479 ; accuracy 0.5837167
epoch 7 : loss 22.305979 ; accuracy 0.58571666
epoch 8 : loss 22.166538 ; accuracy 0.58741665
epoch 9 : loss 22.02943 ; accuracy 0.58931667
epoch 10 : loss 21.894577 ; accuracy 0.59071666
epoch 11 : loss 21.761927 ; accuracy 0.59265
epoch 12 : loss 21.631432 ; accuracy 0.59455
epoch 13 : loss 21.503067 ; accuracy 0.59608334
epoch 14 : loss 21.37677 ; accuracy 0.5978
epoch 15 : loss 21.252499 ; accuracy 0.5996
epoch 16 : loss 21.13023 ; accuracy 0.6014
epoch 17 : loss 21.00993 ; accuracy 0.6031
epoch 18 : loss 20.8916 ; accuracy 0.6048667
epoch 19 : loss 20.775202 ; accuracy 0.60618335
epoch 20 : loss 20.66069 ; accuracy 0.60793334
epoch 21 : loss 20.54799 ; accuracy 0.6095167
